In [1]:
import mysql.connector
import pandas as pd
from mysql.connector import Error
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── CONFIG ────────────────────────────────────────────────────────────────────

DB_CONFIG = {
    "host":     "localhost",
    "user":     "ndadmin",
    "password": "ndADMIN@2025",
    "port":     3306,
}

TARGET_SCHEMAS = ["mobiledoc_apr26"]

NULL_DATE  = "2026-04-11 00:00:00"
NULL_FLAG  = "Y"

In [2]:
def get_tables_from_schema(cursor):
    format_strings = ", ".join(["%s"] * len(TARGET_SCHEMAS))
    cursor.execute(f"""
        SELECT DISTINCT TABLE_SCHEMA, TABLE_NAME
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA IN ({format_strings})
          AND COLUMN_NAME IN ('nd_extracted_at', 'nd_is_active')
    """, TARGET_SCHEMAS)
    return cursor.fetchall()


def column_exists(cursor, schema, table, column):
    cursor.execute("""
        SELECT 1 FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s AND COLUMN_NAME = %s
    """, (schema, table, column))
    return cursor.fetchone() is not None


def migrate_table(cursor, schema, table):
    # ── nd_extracted_at → nd_extracted_date DATETIME ─────────────────────────
    if column_exists(cursor, schema, table, "nd_extracted_at"):
        print(f"  [{schema}.{table}] Renaming nd_extracted_at → nd_extracted_date")
        cursor.execute(f"ALTER TABLE `{schema}`.`{table}` CHANGE COLUMN `nd_extracted_at` `nd_extracted_date` DATETIME")

        cursor.execute(f"""
            UPDATE `{schema}`.`{table}`
            SET `nd_extracted_date` = %s
            WHERE `nd_extracted_date` IS NULL
        """, (NULL_DATE,))
        print(f"  [{schema}.{table}] Updated {cursor.rowcount} NULL rows in nd_extracted_date")
    else:
        print(f"  [{schema}.{table}] nd_extracted_at not found, skipping")

    # ── nd_is_active → nd_ActiveFlag VARCHAR(10) ────────────────────────────
    if column_exists(cursor, schema, table, "nd_is_active"):
        print(f"  [{schema}.{table}] Renaming nd_is_active → nd_ActiveFlag")
        cursor.execute(f"ALTER TABLE `{schema}`.`{table}` CHANGE COLUMN `nd_is_active` `nd_ActiveFlag` VARCHAR(10)")

        cursor.execute(f"""
            UPDATE `{schema}`.`{table}`
            SET `nd_ActiveFlag` = %s
            WHERE `nd_ActiveFlag` IS NULL OR `nd_ActiveFlag` = 'Yes'
        """, (NULL_FLAG,))
        print(f"  [{schema}.{table}] Updated {cursor.rowcount} NULL rows in nd_ActiveFlag")
    else:
        print(f"  [{schema}.{table}] nd_is_active not found, skipping")

def migrate_table_threaded(schema, table):
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
        cursor.execute("SET sql_mode = ''")

        print(f"Processing: {schema}.{table}")
        migrate_table(cursor, schema, table)

        cursor.execute("SET FOREIGN_KEY_CHECKS = 1")
        conn.commit()
        print(f"Done: {schema}.{table}\n")

    except Error as e:
        print(f"Error on {schema}.{table}: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()

In [3]:
USE_MANUAL_LIST = False
MANUAL_TABLE_LIST = [('mobiledoc_apr26_staging', 'doctors'), ('mobiledoc_apr26', 'doctors')]

In [4]:
conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

if USE_MANUAL_LIST:
    tables = MANUAL_TABLE_LIST
    print(f"Using manual list — {len(tables)} table(s)\n")
else:
    tables = get_tables_from_schema(cursor)
    print(f"Discovered {len(tables)} table(s) via INFORMATION_SCHEMA\n")

cursor.close()
conn.close()

Discovered 59 table(s) via INFORMATION_SCHEMA



In [5]:
tables

[('mobiledoc_apr26', 'accounts_assignedto_history'),
 ('mobiledoc_apr26', 'additionalnotes'),
 ('mobiledoc_apr26', 'admin_tblcreationlog'),
 ('mobiledoc_apr26', 'ApptBlocks'),
 ('mobiledoc_apr26', 'ApptRecurrenceWeeklyPattern'),
 ('mobiledoc_apr26', 'assessment_generation_history'),
 ('mobiledoc_apr26', 'assessment_lab_history'),
 ('mobiledoc_apr26', 'assessment_notes_history'),
 ('mobiledoc_apr26', 'assessment_rx_history'),
 ('mobiledoc_apr26', 'AttachedPastOrders'),
 ('mobiledoc_apr26', 'authenticatepwd'),
 ('mobiledoc_apr26', 'batchfaxing'),
 ('mobiledoc_apr26', 'batchfaxinglog'),
 ('mobiledoc_apr26', 'BatchFaxingRules'),
 ('mobiledoc_apr26', 'billingedits'),
 ('mobiledoc_apr26', 'breakglasswarning'),
 ('mobiledoc_apr26', 'CDSS_Logs'),
 ('mobiledoc_apr26', 'cdss_vitals'),
 ('mobiledoc_apr26', 'claim_assignedto_history'),
 ('mobiledoc_apr26', 'claim_assignedTo_historydetail'),
 ('mobiledoc_apr26', 'claimcollectionstatlog'),
 ('mobiledoc_apr26', 'claimicdchangeslog'),
 ('mobiledoc_apr

In [6]:
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(migrate_table_threaded, schema, table): (schema, table) for schema, table in tables}
    for future in as_completed(futures):
        schema, table = futures[future]
        if future.exception():
            print(f"Failed: {schema}.{table} — {future.exception()}")
    
print("Done.")

Processing: mobiledoc_apr26.ApptBlocksProcessing: mobiledoc_apr26.accounts_assignedto_history

Processing: mobiledoc_apr26.additionalnotes
Processing: mobiledoc_apr26.admin_tblcreationlog
  [mobiledoc_apr26.ApptBlocks] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26.additionalnotes] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26.admin_tblcreationlog] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26.accounts_assignedto_history] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26.admin_tblcreationlog] Updated 49 NULL rows in nd_extracted_date
  [mobiledoc_apr26.admin_tblcreationlog] Renaming nd_is_active → nd_ActiveFlag
  [mobiledoc_apr26.admin_tblcreationlog] Updated 50 NULL rows in nd_ActiveFlag
Done: mobiledoc_apr26.admin_tblcreationlog

Processing: mobiledoc_apr26.ApptRecurrenceWeeklyPattern
  [mobiledoc_apr26.ApptRecurrenceWeeklyPattern] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26.ApptRecurrenceWeeklyPat

In [7]:
p0_df = pd.read_csv("/Users/ndaidcnd/Desktop/Air_DEID/airflow-automation/Airflow/input/deid_runner.csv")
tables = set(p0_df['table_name'].to_list())
len(tables), schema

(60, 'mobiledoc_apr26')

In [8]:
schema = 'mobiledoc_apr26_staging'
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(migrate_table_threaded, schema, table): (schema, table) for table in tables}
    for future in as_completed(futures):
        schema, table = futures[future]
        if future.exception():
            print(f"Failed: {schema}.{table} — {future.exception()}")
    
print("Done.")

Processing: mobiledoc_apr26_staging.lablistProcessing: mobiledoc_apr26_staging.edi_invoice

Processing: mobiledoc_apr26_staging.structccmr
Processing: mobiledoc_apr26_staging.cpt_validcodes
  [mobiledoc_apr26_staging.edi_invoice] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26_staging.structccmr] nd_extracted_at not found, skipping
  [mobiledoc_apr26_staging.structccmr] nd_is_active not found, skipping
Done: mobiledoc_apr26_staging.structccmr

Processing: mobiledoc_apr26_staging.encounterdata
  [mobiledoc_apr26_staging.lablist] nd_extracted_at not found, skipping
  [mobiledoc_apr26_staging.cpt_validcodes] nd_extracted_at not found, skipping
  [mobiledoc_apr26_staging.encounterdata] Renaming nd_extracted_at → nd_extracted_date
  [mobiledoc_apr26_staging.lablist] nd_is_active not found, skipping
  [mobiledoc_apr26_staging.cpt_validcodes] nd_is_active not found, skipping
Done: mobiledoc_apr26_staging.lablist

Done: mobiledoc_apr26_staging.cpt_validcodes

Processing: mobile

In [9]:
schema = 'deidentified'
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(migrate_table_threaded, schema, table): (schema, table) for table in tables}
    for future in as_completed(futures):
        schema, table = futures[future]
        if future.exception():
            print(f"Failed: {schema}.{table} — {future.exception()}")
    
print("Done.")

Processing: deidentified.cpt_validcodes
Processing: deidentified.edi_invoice
Processing: deidentified.structccmr
Processing: deidentified.lablist
  [deidentified.lablist] nd_extracted_at not found, skipping
  [deidentified.lablist] nd_is_active not found, skipping
Done: deidentified.lablist

Processing: deidentified.encounterdata
  [deidentified.encounterdata] nd_extracted_at not found, skipping
  [deidentified.encounterdata] nd_is_active not found, skipping
Done: deidentified.encounterdata

Processing: deidentified.cptcode_base
  [deidentified.cpt_validcodes] nd_extracted_at not found, skipping
  [deidentified.edi_invoice] nd_extracted_at not found, skipping
  [deidentified.cptcode_base] nd_extracted_at not found, skipping
  [deidentified.structccmr] nd_extracted_at not found, skipping
  [deidentified.cpt_validcodes] nd_is_active not found, skipping
  [deidentified.edi_invoice] nd_is_active not found, skipping
Done: deidentified.cpt_validcodes

Done: deidentified.edi_invoice

  [deide